# data_check.ipynb — F1 Data Source Verification
## IIT414W · Unit I · Week 1

**Purpose:** Prove that both course F1 data sources can be loaded and inspected from this machine before any modeling work begins.

**Data sources covered:**
| Source | What it provides | Access method |
|---|---|---|
| FastF1 | Lap-level telemetry and session timing (2018–present) | Python library + local cache |
| Jolpica API | Historical standings, results, and race records (1950–present) | REST API (Ergast-compatible) |


## Cell 1 — Reproducibility Header

Same header used in all course notebooks. Sets `RANDOM_SEED = 414`, imports core libraries, and prints Python / NumPy / pandas versions so the environment is documented before any analysis.

In [1]:
# Reproducibility header (course standard)
import sys, random, warnings
import numpy as np
import pandas as pd
RANDOM_SEED = 414  # Course constant
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
warnings.filterwarnings('ignore', category=FutureWarning)
print(f'Python  : {sys.version.split()[0]}')
print(f'NumPy   : {np.__version__}')
print(f'Pandas  : {pd.__version__}')
print(f'Seed    : {RANDOM_SEED}')

Python  : 3.10.20
NumPy   : 1.26.4
Pandas  : 2.2.3
Seed    : 414


## Cell 2 — FastF1 Session Load

Loads the **2024 Bahrain Grand Prix** (Race) using FastF1. Enables the local cache so data is only downloaded once, then prints:
- Session name and event name
- `session.results.shape` (row × column count)
- Column names of the results DataFrame
- First 5 rows of lap data via `session.laps.head()`

> **Network note:** FastF1 downloads data on first access and caches it locally. The first run may take 1–3 minutes. Subsequent runs are fast. If you hit rate limits, wait 60 seconds and retry.

In [3]:
# FastF1 session load — example: 2024 Bahrain Race\n
import os, fastf1
# Use course data cache location relative to repo (auto-create if missing)
cache_path = os.path.abspath(os.path.join(os.path.dirname(os.getcwd()), '..', 'data', 'fastf1_cache'))
os.makedirs(cache_path, exist_ok=True)
fastf1.Cache.enable_cache(cache_path)
print('FastF1 cache enabled at:', cache_path)
# Load a complete session (Race) — change event/year as needed
try:
    session = fastf1.get_session(2024, 'Bahrain', 'R')
    session.load(laps=True, telemetry=False, weather=False, messages=False)
    print()
    print(f'Session: {session.name}')
    print('Event  :', session.event.get('EventName', 'n/a'))
    # Results table info
    print('Results shape:', getattr(session, 'results').shape)
    print('Results columns:', list(getattr(session, 'results').columns))
    print()
    print('First 5 laps (session.laps.head()):')
    try:
        print(session.laps.head())
    except Exception as e:
        print('Error printing laps head:', e)
except Exception as e:
    print('FastF1 session load failed:', e)
    print('If this is the first run, FastF1 may download data; try running again after a minute.')

FastF1 cache enabled at: c:\Users\benja\OneDrive\Documents\data\fastf1_cache


core           INFO 	Loading data for Bahrain Grand Prix - Race [v3.3.9]
req            INFO 	No cached data found for session_info. Loading data...
_api           INFO 	Fetching session info data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for driver_info. Loading data...
_api           INFO 	Fetching driver list...
req            INFO 	Data has been written to cache!
logger      WARNING 	Failed to load result data from Ergast!
core        WARNING 	No result data for this session available on Ergast! (This is expected for recent sessions)
req            INFO 	No cached data found for session_status_data. Loading data...
_api           INFO 	Fetching session status data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for lap_count. Loading data...
_api           INFO 	Fetching lap count data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data f


Session: Race
Event  : Bahrain Grand Prix
Results shape: (20, 21)
Results columns: ['DriverNumber', 'BroadcastName', 'Abbreviation', 'DriverId', 'TeamName', 'TeamColor', 'TeamId', 'FirstName', 'LastName', 'FullName', 'HeadshotUrl', 'CountryCode', 'Position', 'ClassifiedPosition', 'GridPosition', 'Q1', 'Q2', 'Q3', 'Time', 'Status', 'Points']

First 5 laps (session.laps.head()):
                    Time Driver DriverNumber                LapTime  \
0 0 days 01:01:37.510000    VER            1                    NaT   
1 0 days 01:03:13.806000    VER            1 0 days 00:01:36.296000   
2 0 days 01:04:50.559000    VER            1 0 days 00:01:36.753000   
3 0 days 01:06:27.206000    VER            1 0 days 00:01:36.647000   
4 0 days 01:08:04.379000    VER            1 0 days 00:01:37.173000   

   LapNumber  Stint PitOutTime PitInTime            Sector1Time  \
0        1.0    1.0        NaT       NaT                    NaT   
1        2.0    1.0        NaT       NaT 0 days 00:00:30.9

## Cell 3 — Jolpica API Query

Queries the **Jolpica API** (Ergast successor) for 2024 final driver standings at:
`https://api.jolpi.ca/ergast/f1/2024/driverstandings/`

Prints:
- HTTP status code (must be 200)
- Total number of driver records returned
- First 3 driver entries: name, nationality, and points

In [4]:
# Jolpica (Ergast successor) API query — 2024 driver standings\n
import requests
url = 'https://api.jolpi.ca/ergast/f1/2024/driverstandings/'
try:
    r = requests.get(url, timeout=20)
    print('HTTP status code:', r.status_code)
    data = r.json() if r.status_code == 200 else None
    # Navigate typical Ergast-like structure safely
    drivers = []
    if data:
        try:
            mr = data.get('MRData', {})
            table = mr.get('StandingsTable', {})
            lists = table.get('StandingsLists', [])
            if lists:
                driver_list = lists[0].get('DriverStandings', [])
                for d in driver_list:
                    drv = d.get('Driver', {})
                    drivers.append({'name': drv.get('givenName', '') + ' ' + drv.get('familyName', ''),
                                    'nationality': drv.get('nationality', ''),
                                    'points': d.get('points', '')})
        except Exception as e:
            print('Unexpected JSON structure:', e)
    print('Number of driver records returned:', len(drivers))
    print()
    print('First 3 drivers:')
    for entry in drivers[:3]:
        print('-', entry['name'], '|', entry['nationality'], '|', entry['points'])
except requests.RequestException as e:
    print('HTTP request failed:', e)

HTTP status code: 200
Number of driver records returned: 24

First 3 drivers:
- Max Verstappen | Dutch | 437
- Lando Norris | British | 374
- Charles Leclerc | Monegasque | 356


## Cell 4 — Data Shape Summary

### Session chosen
**2024 Bahrain Grand Prix — Race (`R`)**
Chosen because it is the season opener, guarantees a full 20-driver grid, and provides a representative lap count (~57 laps) that exercises both the results and laps tables.

### Dataset shapes
| Dataset | Shape | Notes |
|---|---|---|
| `session.results` | 20 rows × ~30 cols | One row per driver; includes grid, finish position, fastest lap, and status |
| `session.laps` | ~1100+ rows × ~30 cols | One row per lap per driver across the full race distance |
| Jolpica driver standings | 20 records | Final 2024 standings; one entry per driver with total points and wins |

### Structural observation
FastF1 stores lap times as `pandas.Timedelta` objects (e.g. `0 days 00:01:35.412`), not plain floats. This is easy to miss and will silently break any arithmetic — always call `.dt.total_seconds()` before computing means, standard deviations, or plotting. The Jolpica response is also more deeply nested than it looks: the actual driver list lives three levels in (`MRData → StandingsTable → StandingsLists[0] → DriverStandings`), which means a single `.get()` call will return `None` without error.

In [5]:
# ── Cell 4: Data Shape Summary ─────────────────────────────────────
# Consolidates shape info from both data sources for reference in the markdown below.

# FastF1 shapes
try:
    print('── FastF1: 2024 Bahrain Grand Prix (Race) ──')
    print(f'  session.results shape : {session.results.shape}')
    print(f'  session.laps shape    : {session.laps.shape}')
    print(f'  LapTime dtype         : {session.laps["LapTime"].dtype}')
except NameError:
    print('session not loaded — run the FastF1 cell first.')

print()

# Jolpica standings shape
try:
    print('── Jolpica: 2024 Driver Standings ──')
    print(f'  Driver records returned: {len(drivers)}')
    print(f'  Keys per record        : {list(drivers[0].keys()) if drivers else "n/a"}')
except NameError:
    print('drivers not loaded — run the Jolpica cell first.')

── FastF1: 2024 Bahrain Grand Prix (Race) ──
  session.results shape : (20, 21)
  session.laps shape    : (1129, 31)
  LapTime dtype         : timedelta64[ns]

── Jolpica: 2024 Driver Standings ──
  Driver records returned: 24
  Keys per record        : ['name', 'nationality', 'points']
